### Losigtic regresson
### decision tree classifier
### random forest classifier
### tubulate orqali jadval tuzish
# Imtihonga tayyorgarlik

`bank-full.csv` datasetida preprocessing → encoding → scaling → train/test → **4 ta model alohida bo'limlarda** natija → oxirida accuracy jadvali.

In [ ]:
import pandas as pd

df = pd.read_csv("/Users/sherzodbeksharobiddinov/MLFoundation/Sherzod_dev/2-oy_Model_Building/Data/bank-full.csv")
print(df.shape)
df.head()

## 1) Preprocessing: bo'sh qiymatlarni to'ldirish

In [ ]:
def datalarni_toldirsin(df):
    df = df.copy()
    for col in df.columns:
        if df[col].isnull().any():
            if df[col].dtype == "object":
                df[col] = df[col].fillna(df[col].mode()[0])
            else:
                df[col] = df[col].fillna(df[col].mean())
    return df


df = datalarni_toldirsin(df)
print("Null count:", df.isnull().sum().sum())

## 2) Encoding (categorical -> numeric)

In [ ]:
from sklearn.preprocessing import LabelEncoder

def encoding_qilsin(df):
    df = df.copy()
    label_encoder = LabelEncoder()
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = label_encoder.fit_transform(df[col])
    return df


df = encoding_qilsin(df)
df.head()

## 3) Train/Testga ajratish va Scaling

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop("y", axis=1)
y = df["y"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("X_train:", X_train.shape, "X_test:", X_test.shape)

## 4) To'rtta model (har biri alohida)

Quyida har bir model **alohida bo'lim**: `fit` → `predict` → **accuracy** → **classification_report**.
Eng oxirida barcha modellar **accuracy** bo'yicha bir jadvalda qisqacha solishtiriladi.

### 4.1 Logistic Regression


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

logistic_model = LogisticRegression(max_iter=2000, random_state=42)
logistic_model.fit(X_train_scaled, y_train)

y_pred_logistic = logistic_model.predict(X_test_scaled)
acc_logistic = accuracy_score(y_test, y_pred_logistic)
print("=== Logistic Regression ===")
print("Accuracy:", round(acc_logistic, 4))
print(classification_report(y_test, y_pred_logistic))


### 4.2 Decision Tree Classifier


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

decision_tree_model = DecisionTreeClassifier(random_state=42)
decision_tree_model.fit(X_train, y_train)

y_pred_tree = decision_tree_model.predict(X_test)
acc_tree = accuracy_score(y_test, y_pred_tree)
print("=== Decision Tree Classifier ===")
print("Accuracy:", round(acc_tree, 4))
print(classification_report(y_test, y_pred_tree))


### 4.3 Random Forest Classifier


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

random_forest_model = RandomForestClassifier(n_estimators=200, random_state=42)
random_forest_model.fit(X_train, y_train)

y_pred_rf = random_forest_model.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)
print("=== Random Forest Classifier ===")
print("Accuracy:", round(acc_rf, 4))
print(classification_report(y_test, y_pred_rf))


### 4.4 Linear Regression (binary: threshold = 0.5)

Regressor chiqishi uzluksiz; klassifikatsiya uchun `>= 0.5` bilan 0/1 ga aylantiramiz.


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, classification_report

linear_model = LinearRegression()
linear_model.fit(X_train_scaled, y_train)

y_pred_linear_raw = linear_model.predict(X_test_scaled)
y_pred_linear = (y_pred_linear_raw >= 0.5).astype(int)
acc_linear = accuracy_score(y_test, y_pred_linear)
print("=== Linear Regression (threshold=0.5) ===")
print("Accuracy:", round(acc_linear, 4))
print(classification_report(y_test, y_pred_linear))


### 5) Accuracy bo'yicha solishtirish (jadval)


In [ ]:
results = pd.DataFrame(
    {
        "Model": [
            "Logistic Regression",
            "Decision Tree Classifier",
            "Random Forest Classifier",
            "Linear Regression (as classifier)",
        ],
        "Accuracy": [acc_logistic, acc_tree, acc_rf, acc_linear],
    }
).sort_values("Accuracy", ascending=False)

try:
    from tabulate import tabulate

    print(
        tabulate(
            results.reset_index(drop=True),
            headers="keys",
            tablefmt="github",
            floatfmt=".4f",
        )
    )
except ImportError:
    print("(tabulate o'rnatilmagan — yuqoridagi DataFrame yetarli)")

results
